In [1]:
import pandas as pd
import simplejson
import os
import re

In [ ]:
# Resolved from the repo root so this behaves the same in Jupyter (cwd = this directory)
# as under `./run.py language-to-json`.
repo = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(repo, 'src', 'assets', 'data')):
    repo, prev = os.path.dirname(repo), repo
    assert repo != prev, 'run this from inside the wingsearch checkout'

i18n_folder = os.path.join(repo, 'i18n')
result_folder = os.path.join(repo, 'src', 'assets', 'data', 'i18n')
files = [os.path.join(i18n_folder, file) for file in os.listdir(i18n_folder) if file.endswith('.xlsx') and not file.startswith('template')]

In [ ]:
for file in files:
    birds = pd.read_excel(file, sheet_name='Birds', index_col=0).drop(columns="Expansion").to_dict(orient='index')
    bonuses = pd.read_excel(file, sheet_name='Bonuses', index_col=0).drop(columns="Expansion").to_dict(orient='index')
    goals = pd.read_excel(file, sheet_name='Goals', index_col=0).drop(columns="Expansion").to_dict(orient='index')
    other = pd.read_excel(file, sheet_name='Other', index_col=0).to_dict(orient='index')
    parameters = pd.read_excel(file, sheet_name='Parameters', index_col=0).to_dict(orient='index')

    result = {'birds': birds, 'bonuses': bonuses, 'goals': goals, 'other': other, 'parameters': parameters}
    json_file = re.search(r'[\w]+\.xlsx', file).group().replace('.xlsx', '.json')

    if not os.path.exists(result_folder):
        os.makedirs(result_folder)

    # An explicit encoding and newline, so this writes the same bytes on every platform. The
    # committed files were generated on Windows, where text mode turns every \n into \r\n: any
    # regeneration elsewhere rewrote all eleven files with nothing changed in them, which buried
    # whatever had actually changed under 66,000 changed lines.
    with open(os.path.join(result_folder, json_file), 'w', encoding='utf-8', newline='\n') as fp:
        simplejson.dump(result, fp, ignore_nan=True, indent=2)
